In [1]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

from agents import Agent, Runner , set_tracing_disabled
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
load_dotenv()
set_tracing_disabled(True)

In [2]:
local_model=OpenAIChatCompletionsModel(
    model="llama3.2:1b ",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1"
         
) )

In [3]:
from agents import Agent, Runner

# BAD instructions — too vague
bad_agent = Agent(
    model=local_model,
    name="Bad Agent",
    instructions="You are helpful."
)

# GOOD instructions — specific, structured, actionable
good_agent = Agent(
    model=local_model,
    name="Code Reviewer",
    instructions="""
    You are a senior Python code reviewer at a tech company.
    
    YOUR ROLE:
    - Review Python code for bugs, style issues, and performance problems.
    - Rate code quality from 1-10.
    - Suggest specific improvements with corrected code snippets.
    
    YOUR RULES:
    - Always check for: type hints, docstrings, error handling, edge cases.
    - Be constructive, not harsh.
    - If the code is good, say so — don't invent problems.
    - Keep reviews concise (under 200 words).
    
    OUTPUT FORMAT:
    Score: X/10
    Issues: [list]
    Suggestions: [list with code]
    """
)

result = await Runner.run(good_agent, """
Review this code:

def add(x, y):
    return x + y
""")

print(result.final_output)

Score: 8/10

Issues:

1. The function `add` does not include any documentation.
2. It only adds two numbers without considering three or more numbers.

Suggestions:

* Add a docstring to describe what the function `add` does, its parameters, and its return value:
    ```python
def add(x: int, y: int) -> int:
    """
    Adds two integers together.

    Args:
        x (int): The first integer.
        y (int): The second integer.

    Returns:
        int: The sum of x and y.
    """
    return x + y
```
* If the function only takes one non-keyword argument, consider adding a type hint for `x` to ensure it's always an integer:
    ```python
def add(x, y) -> int:
    # ...
```


In [4]:
from agents import Agent, Runner, ModelSettings

# Creative agent — high temperature
creative_agent = Agent(
    model=local_model,
    name="Poet",
    instructions="You write creative, imaginative descriptions. Be vivid and poetic.",
    model_settings=ModelSettings(
        temperature=0.9,       # High = more creative/random
        max_tokens=500,        # Limit output length
    )
)

# Precise agent — low temperature
precise_agent = Agent(
    model=local_model,
    name="Fact Checker",
    instructions="You provide precise, factual answers. No speculation. Cite sources if possible.",
    model_settings=ModelSettings(
        temperature=0.1,       # Low = more deterministic/precise
        max_tokens=300,
    )
)

# Same question, different agents
question = "Describe the Badshahi Mosque in Lahore."

creative_result = await Runner.run(creative_agent, question)
precise_result = await Runner.run(precise_agent, question)

print(" CREATIVE:\n", creative_result.final_output[:300])
print("\n PRECISE:\n", precise_result.final_output[:300])

 CREATIVE:
 The Badshahi Mosque, a majestic testament to the grandeur of Mughal architecture, sprawls across the gentle curves of Lahore like a verdant oasis in the heart of the city. As the sun rises over the sprawling metropolis, its turquoise domes and emerald minarets rise majestically from the dusty landsc

 PRECISE:
 The Badshahi Mosque is a significant Islamic monument located in Lahore, Pakistan. It was built in the early 17th century by Mughal Emperor Shah Jahan as a mausoleum for his wife, Mumtaz Mahal, who died during childbirth in 1631.

The mosque is considered one of the most beautiful examples of Mughal


In [5]:
from pydantic import BaseModel, Field
from agents import Agent, Runner
import json


# ✅ Define schema
class PodcastReview(BaseModel):
    title: str = Field(description="Podcast title")
    host: str = Field(description="Podcast host(s)")
    rating: float = Field(description="Rating from 1.0 to 10.0")
    genre: str = Field(description="Primary genre")
    technical_level: str = Field(description="Beginner, Intermediate, or Advanced")
    summary: str = Field(description="One-sentence summary")
    recommendation: bool = Field(description="Would you recommend this?")


# ✅ FIXED agent (STRICT JSON instructions)
reviewer = Agent(
    model=local_model,
    name="AI Podcast Reviewer",
    instructions=(
        "You are an AI podcast critic.\n\n"
        "IMPORTANT RULES:\n"
        "- Return ONLY valid JSON.\n"
        "- Do NOT write explanations, markdown, or headings.\n"
        "- Output must match this exact format:\n\n"
        "{\n"
        '  "title": "string",\n'
        '  "host": "string",\n'
        '  "rating": 8.5,\n'
        '  "genre": "string",\n'
        '  "technical_level": "Beginner | Intermediate | Advanced",\n'
        '  "summary": "string",\n'
        '  "recommendation": true\n'
        "}\n"
    ),
    output_type=PodcastReview,
)

# ✅ Run agent
result = await Runner.run(reviewer, "Review the podcast 'Lex Fridman Podcast'")

# ✅ SAFE parsing (handles local model mistakes)
try:
    review = result.final_output  # works if JSON is perfect
except Exception:
    # fallback: manually fix
    raw = result.raw_output.strip()
    review = PodcastReview.model_validate_json(raw)


# ✅ Print nicely
print(f"Title:          {review.title}")
print(f"Host:           {review.host}")
print(f"Rating:         {review.rating}/10")
print(f"Genre:          {review.genre}")
print(f"Tech Level:     {review.technical_level}")
print(f"Summary:        {review.summary}")
print(f"Recommended:    {'Yes' if review.recommendation else 'No'}")

Title:          The Lex Fridman Podcast
Host:           Lex Fridman
Rating:         8.5/10
Genre:          Science, Technology, Philosophy
Tech Level:     Advanced
Summary:        In-depth interviews with innovators, scientists, and thinkers.
Recommended:    Yes


In [6]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner
import json


# ✅ Schema
class ProductClassification(BaseModel):
    category: Literal["electronics", "clothing", "food", "books", "other"]
    urgency: Literal["low", "medium", "high"]
    price_range: Literal["budget", "mid", "premium"]
    search_query: str
    confidence: float  # 0.0 - 1.0


# ✅ FIXED agent (STRICT JSON + example)
classifier = Agent(
    model=local_model,
    name="Product Classifier",
    instructions=(
        "You classify customer product requests.\n\n"
        "IMPORTANT:\n"
        "- Return ONLY valid JSON\n"
        "- No bullet points, no markdown, no text\n"
        "- Output MUST match this format:\n\n"
        "{\n"
        '  "category": "electronics | clothing | food | books | other",\n'
        '  "urgency": "low | medium | high",\n'
        '  "price_range": "budget | mid | premium",\n'
        '  "search_query": "string",\n'
        '  "confidence": 0.85\n'
        "}\n\n"
        "Rules:\n"
        "- confidence must be between 0.0 and 1.0\n"
        "- choose closest valid category\n"
    ),
    output_type=ProductClassification,
)

# ✅ Test data
requests = [
    "I need a laptop for my son's school, nothing too expensive",
    "Looking for a birthday gift, maybe a nice novel",
    "URGENT: Need a phone charger, mine just broke!",
]

# ✅ Run safely
for req in requests:
    result = await Runner.run(classifier, req)

    try:
        c = result.final_output  # works if JSON is correct
    except Exception:
        # 🔥 fallback fix for local models
        raw = result.raw_output.strip()

        # remove markdown if present
        raw = raw.replace("```json", "").replace("```", "").strip()

        # try parsing manually
        try:
            c = ProductClassification.model_validate_json(raw)
        except Exception:
            print("\n❌ Failed to parse output:")
            print(raw)
            continue

    print(f"\nRequest: '{req}'")
    print(f"   Category: {c.category} | Urgency: {c.urgency} | Price: {c.price_range}")
    print(f"   Search: '{c.search_query}' | Confidence: {c.confidence:.0%}")


Request: 'I need a laptop for my son's school, nothing too expensive'
   Category: electronics | Urgency: low | Price: mid
   Search: 'laptop' | Confidence: 90%

Request: 'Looking for a birthday gift, maybe a nice novel'
   Category: books | Urgency: medium | Price: mid
   Search: 'birthday novel' | Confidence: 92%

Request: 'URGENT: Need a phone charger, mine just broke!'
   Category: electronics | Urgency: high | Price: budget
   Search: 'phone charger' | Confidence: 90%


In [7]:
from pydantic import BaseModel, Field
from typing import Literal
from agents import Agent, Runner, ModelSettings
import os
from dotenv import load_dotenv

# -----------------------------
# Step 0: Load environment variables
# -----------------------------
load_dotenv()  

# -----------------------------
# Step 1: Define output schema
# -----------------------------
class TicketAnalysis(BaseModel):
    sentiment: Literal["positive", "neutral", "negative", "angry"] = Field(
        description="Customer sentiment"
    )
    priority: Literal["P0-critical", "P1-high", "P2-medium", "P3-low"] = Field(
        description="Ticket priority"
    )
    department: Literal["billing", "technical", "sales", "general"] = Field(
        description="Which department should handle this"
    )
    summary: str = Field(description="One-line summary for the dashboard")
    suggested_response: str = Field(description="Draft first response to customer")

# -----------------------------
# Step 2: Create Agent
# -----------------------------
ticket_analyzer = Agent(
    model=local_model,  # Replace with your model object
    name="Ticket Analyzer",
    instructions="""
You are an AI support ticket analyzer for a SaaS company.

IMPORTANT RULES:
- RETURN ONLY VALID JSON matching the schema.
- DO NOT include bullet points, markdown, headings, or extra text.
- ALL FIELDS MUST BE PRESENT: sentiment, priority, department, summary, suggested_response
- suggested_response must be professional and empathetic.

PRIORITY RULES:
- P0-critical: Service down, data loss, security breach
- P1-high: Feature broken, payment issues, angry customer
- P2-medium: Bug reports, feature requests, how-to questions
- P3-low: General feedback, suggestions, non-urgent inquiries

DEPARTMENT ROUTING:
- billing: Payment, subscription, invoice, refund
- technical: Bugs, errors, API issues, integrations
- sales: Pricing, enterprise plans, demos, upgrades
- general: Everything else

OUTPUT FORMAT EXAMPLE:
{
  "sentiment": "angry",
  "priority": "P0-critical",
  "department": "technical",
  "summary": "API is down and affecting production.",
  "suggested_response": "We are aware of the issue and our team is investigating. Thank you for your patience."
}
""",
    output_type=TicketAnalysis,
    model_settings=ModelSettings(temperature=0.1)
)

# -----------------------------
# Step 3: Simulate tickets
# -----------------------------
tickets = [
    "Your API has been returning 500 errors for 2 hours. Our production is DOWN. Fix this NOW!",
    "Hi, I was charged twice this month. Can you please look into it? Thanks.",
    "Love the product! Any chance you'll add dark mode soon?",
]

# -----------------------------
# Step 4: Run agent safely
# -----------------------------
for ticket in tickets:
    result = await Runner.run(ticket_analyzer, ticket)

    # Handle local model fallback
    try:
        t = result.final_output
    except Exception:
        raw = result.raw_output.strip().replace("```json", "").replace("```", "")
        t = TicketAnalysis.model_validate_json(raw)

    print(f"\n{'='*60}")
    print(f"Ticket: {ticket[:60]}...")
    print(f"Sentiment: {t.sentiment}")
    print(f"Priority:  {t.priority}")
    print(f"Route to:  {t.department}")
    print(f"Summary:   {t.summary}")
    print(f"Response:  {t.suggested_response[:150]}...")


Ticket: Your API has been returning 500 errors for 2 hours. Our prod...
Sentiment: angry
Priority:  P0-critical
Route to:  technical
Summary:   API down and causing production issues.
Response:  We are aware of the issue and our team is working on a fix. Please be patient while we resolve it. Thank you for your understanding....

Ticket: Hi, I was charged twice this month. Can you please look into...
Sentiment: angry
Priority:  P1-high
Route to:  billing
Summary:   Duplicate charge detected.
Response:  We apologize for the inconvenience and are working to resolve this issue as soon as possible. Please contact us if you have any further questions or c...

Ticket: Love the product! Any chance you'll add dark mode soon?...
Sentiment: neutral
Priority:  P1-high
Route to:  sales
Summary:   Customer inquiry about feature availability.
Response:  We appreciate your interest in our product. Our team is working on it, and we will notify you as soon as it's available....
